Initialiser Spark et afficher les couches du Lakehouse

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Jupyter-Iceberg-Explorer").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("Bases disponibles :")
spark.sql("SHOW DATABASES IN lakehouse").show()

print("Couche Gold (daily_revenue) :")
spark.table("lakehouse.gold.daily_revenue").show()

Bases disponibles :
+---------+
|namespace|
+---------+
|   bronze|
|     gold|
|   silver|
+---------+

Couche Gold (daily_revenue) :
+----------+------------+-------------+
|order_date|total_orders|total_revenue|
+----------+------------+-------------+
|2026-11-10|           2|        205.5|
|2026-11-11|           1|        250.0|
+----------+------------+-------------+



 Tester le Time Travel Iceberg
 

In [5]:
# Récupérer les snapshots par ordre chronologique
snapshots = spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM lakehouse.silver.orders_clean.snapshots
    ORDER BY committed_at
""").collect()

first_snapshot_id = snapshots[0]["snapshot_id"]
print(f"Premier snapshot ID : {first_snapshot_id}")

print("=== ÉTAT INITIAL (AVANT UPDATE) ===")
spark.read.option("snapshot-id", first_snapshot_id).table("lakehouse.silver.orders_clean").orderBy("order_id").show()

print("=== ÉTAT ACTUEL (APRÈS UPDATE) ===")
spark.table("lakehouse.silver.orders_clean").orderBy("order_id").show()

Premier snapshot ID : 3496579313912070499
=== ÉTAT INITIAL (AVANT UPDATE) ===
+--------+----------+-----------+------+---------+--------------------+
|order_id|order_date|customer_id|amount|   status|        processed_at|
+--------+----------+-----------+------+---------+--------------------+
|     101|2026-11-10|    CUST-01| 120.5|COMPLETED|2026-11-14 17:50:...|
|     102|2026-11-10|    CUST-02|  85.0|COMPLETED|2026-11-14 17:50:...|
|     104|2026-11-11|    CUST-03| 250.0|COMPLETED|2026-11-14 17:50:...|
+--------+----------+-----------+------+---------+--------------------+

=== ÉTAT ACTUEL (APRÈS UPDATE) ===
+--------+----------+-----------+------+---------+--------------------+
|order_id|order_date|customer_id|amount|   status|        processed_at|
+--------+----------+-----------+------+---------+--------------------+
|     101|2026-11-10|    CUST-01| 120.5|COMPLETED|2026-11-14 17:50:...|
|     102|2026-11-10|    CUST-02|  85.0|COMPLETED|2026-11-14 17:50:...|
|     104|2026-11-11| 

In [6]:
spark.sql("""
    ALTER TABLE lakehouse.silver.orders_clean
    ADD COLUMN payment_method STRING
""")

spark.sql("""
    INSERT INTO lakehouse.silver.orders_clean VALUES
        (105, DATE '2026-11-12', 'CUST-04', 190.00, 'COMPLETED', current_timestamp(), 'CARD')
""")

spark.table("lakehouse.silver.orders_clean").orderBy("order_id").show()

+--------+----------+-----------+------+---------+--------------------+--------------+
|order_id|order_date|customer_id|amount|   status|        processed_at|payment_method|
+--------+----------+-----------+------+---------+--------------------+--------------+
|     101|2026-11-10|    CUST-01| 120.5|COMPLETED|2026-11-14 17:50:...|          NULL|
|     102|2026-11-10|    CUST-02|  85.0|COMPLETED|2026-11-14 17:50:...|          NULL|
|     104|2026-11-11|    CUST-03| 300.0|COMPLETED|2026-11-14 17:50:...|          NULL|
|     105|2026-11-12|    CUST-04| 190.0|COMPLETED|2026-11-14 18:24:...|          CARD|
+--------+----------+-----------+------+---------+--------------------+--------------+

